# Step 4: XGBoost Dual-Objective Matcher
Trains two XGBoost models:
- **Regressor**: predicts detour distance (km)
- **Classifier**: predicts acceptance probability

Includes grid search hyperparameter tuning, 5-fold cross-validation, and saves feature importance + ROC curve plots.

**To run:** Use `Kernel → Restart & Run All` to execute all cells in order.

In [1]:
# Run from project root
import os
if os.path.basename(os.getcwd()) in ['models', 'simulation', 'data']:
    os.chdir('../..')
elif os.path.basename(os.getcwd()) == 'src':
    os.chdir('..')
print('Working directory:', os.getcwd())

# Enable inline plots for notebook display
%matplotlib inline

Working directory: /Users/krunal/Desktop/final-project-team-tle


In [2]:
import pandas as pd
import numpy as np

from src.models.xgb_matching import MultiObjectiveMatcher
print("Imported MultiObjectiveMatcher from src/models/xgb_matching.py")

Imported MultiObjectiveMatcher from src/models/xgb_matching.py


In [3]:
# Run the XGBoost dual-objective matcher
print("Loading simulated candidate pairs...")
df = pd.read_parquet("data/simulated/candidate_pairs.parquet")
matcher = MultiObjectiveMatcher()
X_test, yr_test, yc_test, pred_detour, pred_match = matcher.train(df)

Loading simulated candidate pairs...
Training on 1290 samples with 9 features.
Driver-level split: 1039 train pairs, 251 test pairs.
No driver_id overlap between train and test: True

Running grid search for classifier hyperparameters...


Best classifier params: {'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 50}
Class balance — pos: 460, neg: 579, scale_pos_weight: 1.259

Running 5-fold cross-validation...


Classifier CV Accuracy: 0.9317 (+/- 0.1223)
  Fold scores: [0.6875 1.     1.     0.9856 0.9855]
Regressor CV MAE: 2.8828 (+/- 1.8945)
  Fold scores: [0.8211 0.9249 2.5175 4.8367 5.3139]

Training final models with best hyperparameters...
Regression MAE (Detour km): 2.66
Classification Accuracy: 0.92
Classification Report:
              precision    recall  f1-score   support

           0       0.98      0.89      0.93       149
           1       0.86      0.97      0.91       102

    accuracy                           0.92       251
   macro avg       0.92      0.93      0.92       251
weighted avg       0.93      0.92      0.92       251


Regression RMSE (Detour km): 5.7198
Classification F1: 0.9124
Classification ROC-AUC: 0.9657
Saved figures/feature_importance.png
Saved figures/roc_curve.png

Running feature ablation study...


Accuracy drop when feature removed (higher = more important):
  fare_ratio: +0.0040
  shareability_count: +0.0040
  driver_orig_dist: +0.0000
  overlap_ratio: +0.0000
  time_diff_mins: +0.0000
  driver_approx_fare: +0.0000
  rider_approx_fare: +0.0000
  pickup_hour: +0.0000
  pickup_dow: +0.0000
Saved figures/ablation_study.png

Generating learning curves...
Saved figures/learning_curve.png

Running system-level bipartite matching evaluation...


/Users/krunal/miniforge3/envs/qmutez/lib/python3.10/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
2 fits failed out of a total of 25.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
2 fits failed with the following error:
Traceback (most recent call last):
  File "/Users/krunal/miniforge3/envs/qmutez/lib/python3.10/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/Users/krunal/miniforge3/envs/qmutez/lib/python3.10/site-packages/xgboost/core.py", line 751, in inner_f
    return func(**kwargs)
  File "/Users/krunal/miniforge3/envs/qmutez/lib/python3.10/site-packages/xgboost/sklearn.py", line 1761, in fit
    raise Valu


System-Level Results (Bipartite Matching on Test Set):
  Unique test drivers  : 187
  Optimal matches      : 183
  Match rate           : 97.9%
  Avg driver detour    : 11.88 km
  Avg driver earnings  : +$5.51 per ride
  Avg rider savings    : -$5.51 per ride
  Avg CO2 saved        : 2970 g per match


Saved figures/system_tradeoff_curve.png
